# CampusAI — Agent Loop: free demonstration & screenshots

**המחברת הזאת לא דורשת תשלום או מפתח API.**

1. הרצה ראשונה: הורדת הפרויקט מ־GitHub (אחרי שדחפת את קובצי העדכון).
2. הרצה שנייה: בדיקות + דמו Mock של חיפוש ואישור הזמנה.
3. הרצה שלישית **אופציונלית**: מודל Qwen חינמי אמיתי דרך Hugging Face. היא כבויה כברירת מחדל ומשתמשת במשאבי Colab כשזמינים; אינה דורשת GPU בתשלום.

**חשוב אקדמית:** `ScriptedModel` אינו LLM. הוא רק תחליף קבוע לבדיקת התנהגות מערכת הסוכן. לצורך הדגמת בחירת כלים בידי מודל אמיתי יש להפעיל את הניסוי האופציונלי ולתעד את תוצאותיו האמיתיות (כולל כשלים).


In [ ]:
# Run only AFTER you copied/pushed this patch to GitHub.
import os, shutil, subprocess, sys, json
repo = '/content/campusai_for_agent_demo'
if os.path.exists(repo):
    # This removes ONLY the notebook's own temporary clone, never your PC project.
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/mosahatokay/AI_Campus_Manager.git', repo], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Repository clone ready; current folder:', os.getcwd())


## A. Offline checks — screenshot 1

הבדיקות ב־Mock אינן מחוברות ל־SQL Server או ל־API. שמור צילום של מספר הבדיקות העוברות.


In [ ]:
result = subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=False)
assert result.returncode == 0, 'One or more tests failed — do not claim success.'


## B. Agent loop mock — screenshots 2 & 3

הפלט מראה רצף חיפוש > תשובה > הצעת הזמנה > אישור מזכירה > שמירה, כולל audit ו־tool trace. ההחלטות של `ScriptedModel` ידועות מראש, לכן זה **דמו של הזרימה**, לא ראיה להבנת שפה חופשית.


In [ ]:
subprocess.run([sys.executable, '-m', 'src.demo_agent'], check=True)


## C. OPTIONAL: real, open-source language model — screenshot 4

אפשר להפעיל `RUN_FREE_MODEL = True` בניסיון נפרד, רק אם בוחרים בכך. Qwen רץ על משאבי Colab (CPU או GPU זמין), בלי מנוי API. יש הורדת משקולות מודל, ולכן הוא צורך זמן, זיכרון ותעבורה; אם Colab מגביל את הגישה, **לא צריך לשלם** — נסה שוב בהמשך. אין הבטחה שהמודל הקטן יענה נכון בעברית או יציית לפורמט JSON.

לצורך השוואת איכות המודל ודא בצילום מה הוא באמת בחר והאם הפעולה אושרה/נדחתה. אל תציג את דמו ה־Mock בתור מודל אמיתי.


In [ ]:
RUN_FREE_MODEL = False  # Change to True ONLY if you want to test the real free model
if RUN_FREE_MODEL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45,<5'], check=True)
    import torch
    from src.agent import CampusAgent
    from src.agent_models import LocalQwenModel
    from src.mock_backend import MockBackend
    from src.tools import CampusTools
    model_name = 'Qwen/Qwen2.5-1.5B-Instruct' if torch.cuda.is_available() else 'Qwen/Qwen2.5-0.5B-Instruct'
    print('Model:', model_name, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
    try:
        language_model = LocalQwenModel(model_name=model_name)
        agent = CampusAgent(language_model, CampusTools(MockBackend()), context_mode='compact', max_steps=4)
        result = agent.chat('real-llm-demo', 'מצאי חדר למבחן ביום רביעי 2026-12-02 בין 10:00 ל-12:00, עבור 35 סטודנטים, 20 מחשבים ומקרן.')
        print('Real LLM result:\n', json.dumps(result, ensure_ascii=False, indent=2))
        print('Agent audit:\n', json.dumps(agent.session('real-llm-demo').audit, ensure_ascii=False, indent=2))
        print('Tool trace:\n', json.dumps(agent.tools.trace, ensure_ascii=False, indent=2))
    except Exception as exc:
        print('Real LLM demo could not finish:', type(exc).__name__, str(exc)[:250])
else:
    print('Optional real-model cell skipped. Cost: 0. No model downloaded.')


## תיעוד מוצע להגשה

- צילום **בדיקות**: כל הבדיקות ב־Mock עברו.
- צילום **הצעת שינוי**: `awaiting_confirmation`, מספר ההזמנות לפני אישור נשאר קבוע.
- צילום **לאחר אישור**: `status=ok`, מספר ההזמנות גדל ב־1 וה־trace מציג את הכלים.
- צילום **מודל אמיתי** (אם הופעל): זיהוי המודל, הפעולה שבחר וה־trace. אם התוצאה שגויה, תעד שגיאה כנתון במחקר — לא להסתיר.

Colab Free: משאבים וזמינות GPU אינם מובטחים. אין צורך לשדרג את החשבון כדי להריץ את השלבים הראשונים.
